In [1]:
from Poly import *
from Z import *

In [4]:
class GF:
    def __init__(self, num : int, q : int | None = None, 
                 poly : list | str | PolyMod |None = None,
                 prime : int | None = None, degree : int | None = None):
        # if p and d are given, use those without checking primality for speed
        if prime is not None and degree is not None:
            p, d  = prime, degree
        else: # check that q = p ^ d is a prime power
            prime_pow = prime_power(q)
            if prime_pow == False:
                raise ValueError('finite field has prime power number of elements')
            else:
                p, d = prime_pow
        #-----------------------------------------------------------------------
        # find the irreducible polynomial to mod by
        #-----------------------------------------------------------------------
        #
        # if f(x) is given, check that it is irreducible of degree d
        if type(poly) == list:
            f_x = PolyMod(poly, p)
            assert f_x.deg == d and f_x.is_irred(), (
            'need irreducible polynomial of degree d to get GF(p^d)')
        elif type(poly) == str:
            assert poly in {'rand', 'random', 'r', 'brute', 'b'}, (
                'options besides conway polynomials are random or brute force'
            )
            if poly in {'rand', 'random', 'r'}:
                import random
                while True:
                    # generate a random polynomial of degree d
                    f = PolyMod(random.randint(p**d + 1, 2 * p**d), p)
                    if f.is_irred(): # check if irreducible, otherwise try again
                        mod = f
                        break
            else: # find the least irred poly of degree d by brute force
                for k in range(p**d + 1, 2 * p**d):
                    if k % p == 0: # not irreducible: divisible by x
                        continue
                    f = PolyMod(k, p)
                    if f.is_irred():
                        mod = f
                        break   
        elif type(poly) == PolyMod: # given directly
            mod = poly
        elif poly == None: # by default, lookup the conway polynomial
            import ast
            
            with open('conway_polynomials.txt', 'r') as file:
                for i, line in enumerate(file):
                    if i == 0: 
                        continue
                    if line[-2] == ';': # got to the end
                        raise ValueError(
                            'The conway polynomial for that field is unknown')
                        
                    data = ast.literal_eval(line.rstrip(',\n'))
                    if data[:2] == [p, d]:
                        mod = PolyMod(data[2], p)
                        break
        else: 
            raise ValueError('need an irreducible polynomial')
        #-----------------------------------------------------------------------
        self.id = num % (p ** d)
        self.repr = PolyMod(self.id, p)
        self.mod = mod
        self.char = p
        self.deg = d
        
    def __add__(self, other) -> object:
        assert self.mod == other.mod, 'need same modulus to add'
        result = (self.repr + other.repr) % self.mod
        return GF(result.id, poly=self.mod, prime=self.char, degree=self.deg)
    
    def __mul__(self, other) -> object:
        assert self.mod == other.mod, 'need same modulus to multiply'
        result = (self.repr * other.repr) % self.mod
        return GF(result.id, poly=self.mod, prime=self.char, degree=self.deg)
        
    def __eq__(self, other) -> bool:
        return (self.id == other.id and self.repr == other.repr 
                and self.mod == other.mod)        
    def __repr__(self) -> str:
        return str(self.id)                   


In [13]:
a = GF(7, 3**10)
a.mod, a.repr, a.char, a.deg

([2, 1, 0, 0, 2, 2, 2, 0, 0, 0, 1], [1, 2], 3, 10)

In [9]:
import ast
s = '[1, 2, 3]'
x = ast.literal_eval(s)
x[0]

1

In [ ]:
p, d = 2, 2
with open('conway_polynomials.txt', 'r') as file:
    for i, line in enumerate(file):
        if i == 0: 
            continue
        if line[-2] == ';': # got to the end
            raise ValueError(
                'The conway polynomial for that field is unknown')
            
        data = ast.literal_eval(line.rstrip(',\n'))
        if data[:2] == [p, d]:
            conway = PolyMod(data[2], p)
            break
conway ** 2

[1, 0, 1, 0, 1]